In [ ]:
import pandas as pd
import numpy as np
import arviz as az
from jax import numpy as jnp
from jax.random import PRNGKey
import numpyro
from numpyro import distributions as dist
from numpyro import infer

pd.options.plotting.backend = "plotly"

from summer3.graph import defer, CompartmentValues, Parameter
from summer3.epi import CompartmentalEpiModel, CompartmentMap, \
    CompartmentalModelODE, CategoryData, TransitionFlow, Stratification, \
    strat_data_from_pandas, build_istate, dti_to_epoch

from tb_macro.constants import NAIVE_COMP, LATENT_COMPS, \
    ACTIVE_COMPS, POST_DETECT_COMPS, INFECT_COMPS, AGE_STRATA
from tb_macro.epi import InfectionProcess

In [ ]:
compartments = NAIVE_COMP + LATENT_COMPS + ACTIVE_COMPS + POST_DETECT_COMPS
disease_state = Stratification("disease_state", compartments)
humans = CompartmentMap.new(disease_state)

In [ ]:
age_strat = humans.stratify(Stratification("age", AGE_STRATA))
age_cats = age_strat.categories()
infect_strat = Stratification("infectious", ["low", "high"])
humans.stratify(infect_strat, (disease_state, ["active"]))
clin_strat = Stratification("clinical", ["subclin", "clin"])
humans.stratify(clin_strat, (disease_state, ["active"]))

In [ ]:
times = pd.Index(np.arange(1800.0, 2000.0, 1.0))
epi_model = CompartmentalEpiModel(humans, times)

# Construct infection-related flows
iprocess = defer(InfectionProcess)(age_cats, age_cats, disease_state["active"])
contact_rate = Parameter("contact_rate", 0.2)
freq_dens_exponent = Parameter("freq_dens_exponent", 1.0)
for comp in INFECT_COMPS:
    reinfect_contact_rate = contact_rate * Parameter(f"rel_sus_{comp}", 1.0)
    reinfect_foi = defer(InfectionProcess.process)(iprocess, CompartmentValues, reinfect_contact_rate, freq_dens_exponent)
    reinfect = TransitionFlow(
        f"infect_{comp}", 
        disease_state[comp], 
        disease_state["incipient"], 
        reinfect_foi,
    )
    epi_model.add_flow(reinfect)

# Other flows
contain = TransitionFlow(
    "containment", 
    disease_state["incipient"], 
    disease_state["contained"], 
    Parameter("contain", 0.0),
)
clearance = TransitionFlow(
    "clearance",
    disease_state["contained"],
    disease_state["cleared"],
    Parameter("clearance_rate", 0.0),
)
breakdown = TransitionFlow(
    "breakdown",
    disease_state["contained"],
    disease_state["incipient"],
    Parameter("breakdown_rate", 0.0),
)
progression = TransitionFlow(
    "progression",
    disease_state["incipient"],
    clin_strat["subclin"],
    Parameter("progression", 0.0),
)
increase_infect = TransitionFlow(
    "increase_infectiousness",
    infect_strat["low"],
    infect_strat["high"],
    Parameter("increase_infect", 0.0),
)
decrease_infect = TransitionFlow(
    "decrease_infectiousness",
    infect_strat["high"],
    infect_strat["low"],
    Parameter("decrease_infect", 0.0),
)
clin_dev = TransitionFlow(
    "clinical_develop",
    clin_strat["subclin"],
    clin_strat["clin"],
    Parameter("clinical_development", 0.0),
)
clin_regress = TransitionFlow(
    "clinical_regress",
    clin_strat["clin"],
    clin_strat["subclin"],
    Parameter("clinical_regression", 0.0),
)
self_recovery = TransitionFlow(
    "self_recovery",
    (disease_state["active"], clin_strat["subclin"]),
    disease_state["recovered"],
    Parameter("self_recovery", 0.0),
)

# Add flows to model
epi_model.add_flow(contain)
epi_model.add_flow(clearance)
epi_model.add_flow(breakdown)
epi_model.add_flow(progression)
epi_model.add_flow(increase_infect)
epi_model.add_flow(decrease_infect)
epi_model.add_flow(clin_dev)
epi_model.add_flow(clin_regress)
epi_model.add_flow(self_recovery)

In [ ]:
clin_dev = TransitionFlow(
    "clinical_develop",
    clin_strat["subclin"],
    clin_strat["clin"],
    Parameter("clinical_development", 0.0),
)

In [ ]:
pop_data = pd.Series(index=AGE_STRATA, data=np.array([1000.0, 1500.0, 1500.0]))
base_pops = strat_data_from_pandas(pop_data, age_strat)
init_splits = [0.0] * len(compartments)
init_splits[compartments.index("mtb_naive")] = 0.9
init_splits[compartments.index("active")] = 0.1
pop_splits = [CategoryData(disease_state.categories(), jnp.array((init_splits)))]

epi_model.set_initial_population(base_pops, pop_splits)

In [ ]:
def get_runner(epi_model, params: dict[str, float]):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

In [ ]:
base_params = {
    "contact_rate": 0.2,
    "freq_dens_exponent": 1.0,
    "rel_sus_mtb_naive": 1.0,
    "rel_sus_contained": 1.0, 
    "rel_sus_cleared": 1.0, 
    "rel_sus_recovered": 1.0, 
    "contain": 0.0,
    "clearance_rate": 0.0,
    "breakdown_rate": 0.0,
    "progression": 0.0,
    "increase_infect": 0.0,
    "decrease_infect": 0.0,
    "clinical_development": 0.0,
    "clinical_regression": 0.0,
    "self_recovery": 0.0,
}
runner, istate = get_runner(epi_model, base_params)
results = epi_model.run(base_params)

In [ ]:
inf_target = (
    results["flows"]["infect_mtb_naive"]
    .sum(to_dims="time")
    .to_pandas_df()
    .rolling(7)
    .sum()[7:60:7]
)["data"]
inf_target_fuzzy = inf_target * np.exp(
    np.random.normal(scale=0.01, size=len(inf_target))
)
inf_target_fuzzy.plot()

In [ ]:
def get_derived_results(params):
    results = epi_model.run(params)  # runner.run(istate.data, params)
    inf_flow = results["flows"]["infect_mtb_naive"]
    weekly_target = (
        inf_flow.sum(to_dims="time").rolling(7, jnp.sum).query(time=inf_target.index)
    )
    return weekly_target

In [ ]:
priors = {
    "contact_rate": dist.Uniform(0.001, 0.5),
}

In [ ]:
def model():
    params = base_params | {k: numpyro.sample(k, v) for k, v in priors.items()}
    weekly_modelled = get_derived_results(params)

    ll = dist.Poisson(inf_target_fuzzy.to_numpy()).log_prob(weekly_modelled.data)
    numpyro.factor("ll", ll)

In [ ]:
kernel = infer.NUTS(model)
mcmc = infer.MCMC(kernel, num_warmup=200, num_samples=200, num_chains=4)
k = PRNGKey(0)
mcmc.run(k)

In [ ]:
idata = az.from_numpyro(mcmc)

In [ ]:
az.summary(idata)

In [ ]:
az.plot_posterior(idata)

In [ ]:
az.plot_trace(idata, compact=False)